In [ ]:
import sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

In [ ]:
df = pd.read_csv("appointments_clean.csv")

In [ ]:
appointment_ids = df["appointment_id"]

In [ ]:
df["reminder_sent"] = df["reminder_sent"].map({"Yes": 1, "No": 0})

In [ ]:
feature_cols = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows",
    "reminder_sent",
    "booking_channel",
    "visit_type",
]

In [ ]:
X_base = df[feature_cols].copy()

In [ ]:
X = pd.get_dummies(X_base, columns=["booking_channel", "visit_type"], drop_first=False)

In [ ]:
y = df["no_show"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
print(f"Training set size: {X_train.shape[0]} rows")
print(f"Test set size: {X_test.shape[0]} rows")
print(f"Number of no-shows in the test set: {y_test.sum()}")

In [ ]:
baseline_accuracy = (y_test == 0).mean()
print(f"Baseline accuracy (always predicting 'Attended'): {baseline_accuracy:.4f}")

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_s, y_train)

In [ ]:
y_pred_lr = lr_model.predict(X_test_s)
y_prob_lr = lr_model.predict_proba(X_test_s)[:, 1]

In [ ]:
print("--- Logistic Regression Test Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

In [ ]:
print("\n--- Feature Coefficients ---")
for col, coef in zip(X.columns, lr_model.coef_[0]):
  print(f"{col}: {coef:.4f}")

In [ ]:
reminder_idx = list(X.columns).index("reminder_sent")
raw_coef = lr_model.coef_[0][reminder_idx]
feature_std = scaler.scale_[reminder_idx]

In [ ]:
exp_coef = np.exp(raw_coef)
true_reminder_effect = np.exp(raw_coef / feature_std)

In [ ]:
print(
    f"\nRaw model coefficient for reminder_sent: {raw_coef:.4f} -> exp(coef):"
    f" {exp_coef:.4f}"
)
print(
    f"True effect of reminder (divided by feature std {feature_std:.4f}):"
    f" {true_reminder_effect:.4f}"
)

In [ ]:
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

In [ ]:
train_acc_full = accuracy_score(y_train, dt_full.predict(X_train))
test_acc_full = accuracy_score(y_test, dt_full.predict(X_test))

In [ ]:
dt_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_depth3.fit(X_train, y_train)

In [ ]:
train_acc_d3 = accuracy_score(y_train, dt_depth3.predict(X_train))
test_acc_d3 = accuracy_score(y_test, dt_depth3.predict(X_test))

In [ ]:
print(
    f"Unbounded Tree -> Train Acc: {train_acc_full:.4f} | Test Acc:"
    f" {test_acc_full:.4f}"
)

In [ ]:
print(
    f"Depth-3 Tree   -> Train Acc: {train_acc_d3:.4f} | Test Acc:"
    f" {test_acc_d3:.4f}"
)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
print(f"Random Forest Accuracy:  {accuracy_score(y_test, rf_pred):.4f}")
print(f"Random Forest Recall:    {recall_score(y_test, rf_pred):.4f}")
print(f"Random Forest ROC-AUC:   {roc_auc_score(y_test, rf_prob):.4f}")

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
print("\nTop 3 Feature Importances:")
print(importances.nlargest(3))

In [ ]:
comparison_data = {
    "Model": ["Logistic Regression", "Depth-3 Decision Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        test_acc_d3,
        accuracy_score(y_test, rf_pred),
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, dt_depth3.predict(X_test)),
        recall_score(y_test, rf_pred),
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(
            y_test, dt_depth3.predict_proba(X_test)[:, 1]
        ),  # evaluation metric
        roc_auc_score(y_test, rf_prob),
    ],
}
comparison_df = pd.DataFrame(comparison_data)
print("\n", comparison_df.to_string(index=False))

In [ ]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

In [ ]:
param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc",
)
grid_search.fit(X_train, y_train)

In [ ]:
print(f"Best Parameters (GridSearchCV): {grid_search.best_params_}")
print(f"Best CV ROC-AUC (Tuned RF): {grid_search.best_score_:.4f}")

In [ ]:
cv_lr = cross_val_score(
    lr_pipeline, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

In [ ]:
cv_dt3 = cross_val_score(
    DecisionTreeClassifier(max_depth=3, random_state=42),
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc",
).mean()

In [ ]:
cv_rf = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc",
).mean()
cv_tuned_rf = grid_search.best_score_

In [ ]:
print("\n--- 5-Fold Cross-Validated ROC-AUC Comparison ---")
print(f"Logistic Regression: {cv_lr:.4f}")
print(f"Depth-3 Tree:        {cv_dt3:.4f}")
print(f"Standard RF:         {cv_rf:.4f}")
print(f"Tuned RF:            {cv_tuned_rf:.4f}")

In [ ]:
winner_model = grid_search.best_estimator_

In [ ]:
oof_probs = cross_val_predict(
    winner_model, X_train, y_train, cv=5, method="predict_proba"
)[:, 1]

In [ ]:
for thresh in [0.5, 0.35]:
  preds = (oof_probs >= thresh).astype(int)
  print(f"\nThreshold: {thresh}")
  print(f"  Precision: {precision_score(y_train, preds, zero_division=0):.4f}")
  print(f"  Recall:    {recall_score(y_train, preds):.4f}")
  print(f"  F1 Score:  {f1_score(y_train, preds, zero_division=0):.4f}")
  print(f"  Flagged:   {preds.sum()} out of {len(y_train)}")

In [ ]:
chosen_threshold = 0.35

In [ ]:
winner_model.fit(X_train, y_train)
test_probs = winner_model.predict_proba(X_test)[:, 1]
test_preds_final = (test_probs >= chosen_threshold).astype(int)

In [ ]:
print("\n--- Final Test Evaluation (Once Only) ---")
print(f"Precision: {precision_score(y_test, test_preds_final, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, test_preds_final):.4f}")
print(f"Flagged:   {test_preds_final.sum()} out of {len(X_test)}")
print(f"ROC-AUC:   {roc_auc_score(y_test, test_probs):.4f}")

In [ ]:
test_results_df = pd.DataFrame({
    "appointment_id": appointment_ids.loc[X_test.index],
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_preds_final,
})
test_results_df.to_csv("test_risk_scores.csv", index=False)
print("\nSaved test_risk_scores.csv successfully.")